# Programmatic Data Dictionary Analysis

This notebook turns the CMS data dictionary into an executable validation artifact. It retrieves a limited sample from the official **Medicare Inpatient Hospitals — by Provider and Service** API, connects each returned field to its analytical meaning, and profiles the variables according to their semantic role.

The notebook is for schema understanding and access validation. It does **not** acquire the complete dataset, persist API records, or produce substantive hospital findings.

## 1. Source and analytical grain

Official dataset: [CMS Medicare Inpatient Hospitals — by Provider and Service](https://data.cms.gov/provider-summary-by-type-of-service/medicare-inpatient-hospitals/medicare-inpatient-hospitals-by-provider-and-service)  
Official definitions: [CMS data dictionary](https://data.cms.gov/resources/medicare-inpatient-hospitals-by-provider-and-service-data-dictionary-0)

The expected observation is one rendering provider and MS-DRG combination within the selected reporting year. The API response does not include a reporting-year column, so the year must later be attached from verified source metadata during acquisition.

In [ ]:
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)

## 2. Executable semantic dictionary

The metadata below is deliberately encoded as data rather than prose alone. This makes it possible to test whether expected fields are present, assign analytical types, and generate consistent profiling rules.

In [ ]:
VARIABLES = [
    {"variable": "Rndrng_Prvdr_CCN", "concept": "Rendering Provider CCN", "role": "identifier", "meaning": "CMS Certification Number identifying the institutional provider.", "caution": "Preserve as text; leading zeros are meaningful and provider names are not substitute keys."},
    {"variable": "Rndrng_Prvdr_Org_Name", "concept": "Rendering Provider Name", "role": "label", "meaning": "Human-readable organizational name associated with the billing provider.", "caution": "Branding and spelling can change, so the name should not be used as a relational key."},
    {"variable": "Rndrng_Prvdr_St", "concept": "Rendering Provider Street Address", "role": "geography", "meaning": "Reported street address of the provider's physical location.", "caution": "Treat as descriptive text and expect formatting variation across releases."},
    {"variable": "Rndrng_Prvdr_City", "concept": "Rendering Provider City", "role": "geography", "meaning": "Municipality in which the provider is physically located.", "caution": "City names are not standardized geographic identifiers by themselves."},
    {"variable": "Rndrng_Prvdr_State_FIPS", "concept": "Rendering Provider State FIPS Code", "role": "geographic_code", "meaning": "Standardized FIPS code for the provider's state.", "caution": "Preserve as fixed-width text and reconcile with the state abbreviation."},
    {"variable": "Rndrng_Prvdr_Zip5", "concept": "Rendering Provider ZIP Code", "role": "geographic_code", "meaning": "Five-digit postal code for the provider location.", "caution": "A ZIP code is not a hospital market, service area, or numeric measure."},
    {"variable": "Rndrng_Prvdr_State_Abrvtn", "concept": "Rendering Provider State Abbreviation", "role": "geographic_code", "meaning": "Two-letter abbreviation for the provider's state.", "caution": "Check consistency with the FIPS code before geographic aggregation."},
    {"variable": "Rndrng_Prvdr_RUCA", "concept": "Rendering Provider RUCA Code", "role": "classification", "meaning": "Rural-Urban Commuting Area classification assigned from the provider ZIP code.", "caution": "Treat as categorical; it describes geographic context, not hospital complexity."},
    {"variable": "Rndrng_Prvdr_RUCA_Desc", "concept": "Rendering Provider RUCA Description", "role": "label", "meaning": "Readable description associated with the RUCA code.", "caution": "Use with the RUCA code and avoid causal conclusions from the classification alone."},
    {"variable": "DRG_Cd", "concept": "MS-DRG Code", "role": "classification", "meaning": "Code grouping inpatient stays with related clinical characteristics and expected resource use.", "caution": "Preserve as text and pair with reporting year because classifications can change."},
    {"variable": "DRG_Desc", "concept": "MS-DRG Description", "role": "label", "meaning": "Readable clinical and severity-oriented description of the MS-DRG.", "caution": "Use for interpretation, not as a stable key."},
    {"variable": "Tot_Dschrgs", "concept": "Total Discharges", "role": "count", "meaning": "Published inpatient discharge volume for the provider and MS-DRG.", "caution": "Rows with 10 or fewer discharges are suppressed; absence therefore does not establish zero activity."},
    {"variable": "Avg_Submtd_Cvrd_Chrg", "concept": "Average Submitted Covered Charge", "role": "currency", "meaning": "Average provider charge for Medicare-covered services associated with the DRG discharges.", "caution": "A charge is not payment, cost, realized revenue, or margin."},
    {"variable": "Avg_Tot_Pymt_Amt", "concept": "Average Total Payment", "role": "currency", "meaning": "Average payment including applicable IPPS components, beneficiary cost sharing, and qualifying third-party payments.", "caution": "Do not interpret this measure as Medicare's share alone."},
    {"variable": "Avg_Mdcr_Pymt_Amt", "concept": "Average Medicare Payment", "role": "currency", "meaning": "Average amount paid by Medicare, including applicable IPPS payment adjustments but excluding beneficiary and third-party amounts.", "caution": "Observed differences are screening signals, not evidence of efficiency or profitability."},
]

dictionary_df = pd.DataFrame(VARIABLES)
dictionary_df

## 3. Retrieve a limited official API sample

Only 100 records are requested. The response remains in memory and is not written to the repository. Change `SAMPLE_SIZE` only for schema exploration; full acquisition should use a separate, controlled process with pagination and provenance checks.

In [ ]:
API_ENDPOINT = "https://data.cms.gov/data-api/v1/dataset/690ddc6c-2767-4618-b277-420ffb2bf27c/data"
SAMPLE_SIZE = 100

def fetch_cms_sample(endpoint: str, size: int = 100) -> pd.DataFrame:
    """Return a small CMS API sample without persisting it locally."""
    if not 1 <= size <= 500:
        raise ValueError("For this exploratory notebook, size must be between 1 and 500.")

    url = f"{endpoint}?{urlencode({'size': size})}"
    request = Request(url, headers={"Accept": "application/json", "User-Agent": "healthcare-operations-analytics/1.0"})
    with urlopen(request, timeout=30) as response:
        status = response.status
        payload = json.load(response)

    if status != 200:
        raise RuntimeError(f"CMS API returned HTTP {status}.")

    records = payload.get("data", payload) if isinstance(payload, dict) else payload
    if not isinstance(records, list) or not records:
        raise ValueError("The API response did not contain a non-empty record list.")

    print(f"HTTP {status}; received {len(records)} records; authentication was not requested.")
    return pd.DataFrame.from_records(records)

sample_raw = fetch_cms_sample(API_ENDPOINT, SAMPLE_SIZE)
sample_raw.head(3)

## 4. Reconcile the live schema with the dictionary

A data dictionary is useful only when it corresponds to the source actually returned. This test identifies expected, missing, and unexpected fields without silently discarding schema changes.

In [ ]:
expected_fields = dictionary_df["variable"].tolist()
actual_fields = sample_raw.columns.tolist()

schema_audit = pd.DataFrame({
    "expected_variable": expected_fields,
    "returned_by_api": [field in actual_fields for field in expected_fields],
})
unexpected_fields = sorted(set(actual_fields) - set(expected_fields))

display(schema_audit)
print("Unexpected API fields:", unexpected_fields or "None")

missing_fields = schema_audit.loc[~schema_audit["returned_by_api"], "expected_variable"].tolist()
if missing_fields:
    raise AssertionError(f"Expected CMS fields were not returned: {missing_fields}")

## 5. Profile variables according to meaning

API values may arrive as strings even when they represent counts or money. The notebook creates a separate analytical copy for type conversion, preserving the original response for comparison. Identifier and geographic codes remain text because arithmetic on them has no substantive meaning.

In [ ]:
sample = sample_raw.copy()
numeric_fields = dictionary_df.loc[dictionary_df["role"].isin(["count", "currency"]), "variable"]

for field in numeric_fields:
    cleaned = sample[field].astype("string").str.replace(r"[$,]", "", regex=True).str.strip()
    sample[field] = pd.to_numeric(cleaned, errors="coerce")

def examples(series: pd.Series, limit: int = 3) -> str:
    values = series.dropna().astype(str).drop_duplicates().head(limit).tolist()
    return " | ".join(values)

profile_rows = []
for item in VARIABLES:
    field = item["variable"]
    series = sample[field]
    profile_rows.append({
        "variable": field,
        "role": item["role"],
        "pandas_dtype": str(series.dtype),
        "non_null": int(series.notna().sum()),
        "null_pct": round(series.isna().mean() * 100, 2),
        "distinct_in_sample": int(series.nunique(dropna=True)),
        "examples": examples(series),
    })

profile_df = pd.DataFrame(profile_rows)
profile_df

In [ ]:
quantitative_summary = (
    sample[numeric_fields.tolist()]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .T
    .rename_axis("variable")
)
quantitative_summary

In [ ]:
format_checks = pd.DataFrame([
    {"check": "Provider CCN is six digits", "valid_in_sample": sample_raw["Rndrng_Prvdr_CCN"].astype("string").str.fullmatch(r"\d{6}", na=False).all()},
    {"check": "State FIPS is two digits", "valid_in_sample": sample_raw["Rndrng_Prvdr_State_FIPS"].astype("string").str.fullmatch(r"\d{2}", na=False).all()},
    {"check": "ZIP is five digits", "valid_in_sample": sample_raw["Rndrng_Prvdr_Zip5"].astype("string").str.fullmatch(r"\d{5}", na=False).all()},
    {"check": "State abbreviation is two letters", "valid_in_sample": sample_raw["Rndrng_Prvdr_State_Abrvtn"].astype("string").str.fullmatch(r"[A-Z]{2}", na=False).all()},
    {"check": "Discharges are positive integers after conversion", "valid_in_sample": ((sample["Tot_Dschrgs"] > 0) & (sample["Tot_Dschrgs"] % 1 == 0)).all()},
])
format_checks

## 6. Interpretation map

The following map connects each family of variables to the question it can support and, equally importantly, to conclusions it cannot justify on its own.

In [ ]:
interpretation_map = pd.DataFrame([
    {"variable_family": "Provider identifiers and labels", "variables": "CCN, organization name", "supports": "Hospital-level grouping and readable reporting", "does_not_establish": "Stable ownership, system membership, or facility equivalence across time"},
    {"variable_family": "Geography", "variables": "Address, city, FIPS, ZIP, state, RUCA", "supports": "Location context and defensible geographic groupings", "does_not_establish": "Hospital market boundaries or operational peer comparability"},
    {"variable_family": "Service classification", "variables": "DRG code and description", "supports": "Comparison of clinically related inpatient categories", "does_not_establish": "Service-line profitability, quality, or clinical appropriateness"},
    {"variable_family": "Activity", "variables": "Total discharges", "supports": "Published service volume above the CMS suppression threshold", "does_not_establish": "Unique patients, capacity, length of stay, or total hospital demand"},
    {"variable_family": "Financial measures", "variables": "Average charge, total payment, Medicare payment", "supports": "Payment-pattern screening within comparable DRGs", "does_not_establish": "Cost, margin, profitability, efficiency, or the cause of payment variation"},
])
interpretation_map

## 7. What this notebook verifies

If all cells run successfully, the notebook verifies that the official endpoint is reachable without authentication, returns records, and exposes the fields described in the project dictionary. It also demonstrates reproducible type conversion and sample-level validation.

It does **not** verify full-file row counts, final data types, null patterns, uniqueness, file size, checksum, multi-year comparability, or analytical findings. Those controls belong to the later acquisition and data-quality workflow.